# P5 — Juntándolo todo

Hora de *demostrarlo*. Tres modelos, la misma tarea de recuperación de largo alcance:

| modelo | alcance | pesos |
|---|---|---|
| **GAT** | 1 salto | aprendidos |
| **walkraw** | multi-salto (álgebra) | fijos = conteo |
| **WalkAttention** | multi-salto (álgebra) | aprendidos |

**5 figuras:** el alcance de las tres arquitecturas · accuracy final · curvas de entrenamiento · pesos de
atención entrenados · la flecha ganadora sobre el grafo.

> Versión pequeña y rápida (~2 min CPU). Resultado completo de 5 seeds: `WalkAttention = 1.000 ± 0.000`.

## Figura 1 — qué puede alcanzar cada modelo

En un diamante pequeño: **GAT** solo ve aristas directas (1 salto). **walkraw** y **WalkAttention** ven todo el
soporte multi-salto que aporta el álgebra de caminos. Alcanzar lejos es necesario — pero, como veremos, no
suficiente.

In [ ]:
from oversquash import viz
dia = [(0,1),(0,2),(1,3),(2,3)]; pos = {0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}
viz.explain_architectures([
    {'title':'GAT: solo 1 salto', 'edges':dia, 'pos':pos, 'n_nodes':4, 'src':0, 'dst':3,
     'reach_edges':[(0,1),(0,2)], 'color':'#fca5a5'},
    {'title':'walkraw: multi-salto, fijo', 'edges':dia, 'pos':pos, 'n_nodes':4, 'src':0, 'dst':3,
     'reach_edges':dia, 'color':'#f59e0b'},
    {'title':'WalkAttention: multi-salto, aprendido', 'edges':dia, 'pos':pos, 'n_nodes':4, 'src':0, 'dst':3,
     'reach_edges':dia, 'color':'#10b981'},
], suptitle='alcance de cada modelo (aristas resaltadas = lo que agrega)')

In [ ]:
import torch, torch.nn.functional as F, numpy as np, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model
K, M, DEPTH = 5, 4, 3   # azar = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]

def entrenar(m, tr, va, fwd, epochs=70, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e+1)/warmup if e < warmup
                                            else 0.5*(1+math.cos(math.pi*(e-warmup)/max(1,epochs-warmup))))
    curva = []
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_ = fwd(m, b)
            loss = F.cross_entropy(lg, b.y, ignore_index=-100); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        m.eval(); a = []
        with torch.no_grad():
            for b in va:
                lg,_ = fwd(m, b); mm = b.root_mask
                a.append((lg[mm].argmax(-1) == b.y[mm]).float().mean().item())
        curva.append(float(np.mean(a)))
    return curva

def correr(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name == 'gat':
        tr = PyGLoader(data[:1200], batch_size=128, shuffle=True); va = PyGLoader(data[1200:], batch_size=128)
        fwd = lambda m,b: m(b.x, b.edge_index, getattr(b,'batch',None))
    elif name == 'walkraw':
        tf = AttachWalkOperators(n_layers=nl); data = [tf(d) for d in data]
        tr = DataLoader(data[:1200], batch_size=128, shuffle=True, collate_fn=collate_walk_operators)
        va = DataLoader(data[1200:], batch_size=128, collate_fn=collate_walk_operators)
        fwd = lambda m,b: m(b.x, b.edge_index, getattr(b,'batch',None), walk_raw=b.walk_raw)
    else:
        tf = AttachWalkMasks(n_layers=nl); data = [tf(d) for d in data]
        tr = DataLoader(data[:1200], batch_size=128, shuffle=True, collate_fn=collate_walk_masks)
        va = DataLoader(data[1200:], batch_size=128, collate_fn=collate_walk_masks)
        fwd = lambda m,b: m(b.x, b.edge_index, getattr(b,'batch',None), walk_masks=b.walk_masks)
    m = build_model(name, data[0].x.size(-1), 16, data[0].num_classes, nl, heads=4)
    return m, entrenar(m, tr, va, fwd)

curvas, modelos = {}, {}
for name in ['gat', 'walkraw', 'walkattn']:
    modelos[name], curvas[name] = correr(name)
    print(f'{name:10s}: accuracy final = {curvas[name][-1]:.3f}')

## Figura 2 — accuracy final

GAT cerca del azar (1 salto no cruza el cuello de botella); walkraw sobre el azar pero estancado (alcanza, no
selecciona); WalkAttention el más alto — normalmente perfecto.

In [ ]:
viz.plot_model_bars(['GAT', 'walkraw', 'WalkAttention'],
                    [curvas['gat'][-1], curvas['walkraw'][-1], curvas['walkattn'][-1]],
                    'recuperación de largo alcance (profundidad 3)', ylabel='accuracy', chance=1/K)

## Figura 3 — curvas de entrenamiento

Qué tan rápido aprende cada modelo. WalkAttention sube hasta arriba; los otros se estancan.

In [ ]:
viz.plot_training_curves({'GAT': curvas['gat'], 'walkraw': curvas['walkraw'],
                          'WalkAttention': curvas['walkattn']},
                         'accuracy vs época', xlabel='época', ylabel='accuracy', chance=1/K)

## Figura 4 — qué aprendió la atención entrenada

Pesos aprendidos hacia el objetivo del WalkAttention **entrenado**. A diferencia de los pesos planos de
walkraw, estos se concentran en la fuente que tiene la respuesta — ese es todo el punto.

In [ ]:
from oversquash.ideal_bridge import walk_operators
from torch_geometric.utils import softmax as pyg_softmax
d0 = ds(0)[0]
raw,_ = walk_operators(d0.edge_index.numpy(), d0.num_nodes, max_length=DEPTH+1)
N = d0.num_nodes; t = int(d0.root_mask.nonzero()[0])
mask = torch.from_numpy((raw[DEPTH] > 0).T.astype('float32')).to_sparse().coalesce()
tgt, src = mask.indices()[0], mask.indices()[1]
L0 = modelos['walkattn'].layers[0]
with torch.no_grad():
    q = L0.q(d0.x).view(N, L0.n_heads, -1); k = L0.k(d0.x).view(N, L0.n_heads, -1)
    logit = (q[tgt]*k[src]).sum(-1).mean(-1) * L0.scale
    alpha = pyg_softmax(logit, tgt, num_nodes=N).numpy()
into_t = (tgt.numpy() == t)
fixed = np.ones(into_t.sum())/into_t.sum()
viz.plot_fixed_vs_learned(src.numpy()[into_t], fixed, alpha[into_t],
                          'entrenado: fijo (walkraw) vs aprendido (WalkAttention)',
                          legend=('fijo (walkraw)', 'aprendido (WalkAttention)'))

## Figura 5 — la atención ganadora, sobre el grafo

La atención entrenada hacia el objetivo, dibujada como grosor de flecha desde cada fuente. El modelo aprendió
a hacer gruesa una flecha — encontró la fuente con la respuesta. **El álgebra de caminos define una atención
sparse multi-salto que mitiga el over-squashing. QED.**

In [ ]:
ws = alpha[into_t]
Kn = len(ws)
edges = [(i, Kn) for i in range(Kn)]
weight_set = {(i, Kn): float(w) for i, w in enumerate(ws)}
ypos = np.linspace(1, -1, Kn)
gpos = {i: (0.0, float(ypos[i])) for i in range(Kn)}; gpos[Kn] = (1.6, 0.0)
viz.explain_attention_on_graph(edges, gpos, [weight_set], Kn + 1,
                               src_nodes=list(range(Kn)), dst=Kn,
                               titles=['atención entrenada hacia el objetivo'],
                               suptitle='el modelo se enfoca en la fuente correcta')
print('Listo. Del over-squashing a una atención sparse multi-salto, demostrado de punta a punta.')